1: Instalação de Dependências

In [0]:
%pip install transformers torch tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.0/146.0 MB 66.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 553.3/553.3 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 798.7/798.7 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.2/536.2 kB 7.2 MB/s eta 0:00:00
  Attempting uninstall: click
    Found existing installation: click 8.1.7
    Not uninstalling click at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-37f2e4a6-2e56-43b1-b2a3-098873355dc6
    Can't uninstall 'click'. No fil

2: Imports e Configurações

In [0]:
import os
import re
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from transformers import pipeline

# Configuração de caminhos usando Volumes do Unity Catalog
# Note que para Pandas acessar o Volume, usamos o caminho direto do sistema de arquivos
input_path = '/Volumes/workspace/voc/voc/voc_gold_2025.csv'
output_path = '/Volumes/workspace/voc/voc/voc_gold_processed_2025.csv'

/local_disk0/.ephemeral_nfs/envs/pythonEnv-37f2e4a6-2e56-43b1-b2a3-098873355dc6/lib/python3.12/site-packages/torch/_vmap_internals.py:9: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  from torch.utils._pytree import _broadcast_to_and_flatten, tree_flatten, tree_unflatten


3: Carregamento dos Dados

In [0]:
if not os.path.exists(input_path):
    raise FileNotFoundError(f"O arquivo não foi encontrado no Volume: {input_path}")

df = pd.read_csv(input_path)

# Limpeza inicial
df['data'] = pd.to_datetime(df['data'], errors='coerce')
df['origem'] = df['origem'].astype(str).str.strip()
df['usuario'] = df['usuario'].astype(str).str.strip()
df['mensagem'] = df['mensagem'].astype(str).fillna('')

print(f"Dados carregados: {df.shape[0]} linhas.")
df.head()

Dados carregados: 2200 linhas.


,origem,usuario,mensagem,data,sentimentalidade,categoria
0,BOT,lucas.dias,"Boa, o erro de sincronizacao sumiu.",2025-01-01 05:17:04,NEUTRA,BUGS
1,BOT,joao.correa,"Estou sem retorno ha dias, isso prejudica o tr...",2025-01-01 08:05:13,NEGATIVA,SUPORTE/ATENDIMENTO
2,CSAT,rafael.ribeiro,Tem trilha de treinamento para novos usuarios?,2025-01-01 13:32:22,NEUTRA,TREINAMENTO/ONBOARDING
3,NPS,thiago.lima,A API esta retornando 401 mesmo com token valido.,2025-01-01 16:45:27,NEGATIVA,API/AUTENTICACAO
4,CSAT,bruno.souza,"Estou sem retorno ha dias, isso prejudica o tr...",2025-01-01 23:54:35,NEGATIVA,SUPORTE/ATENDIMENTO


Inicialização do Modelo (Hugging Face) - Voce pode usar aqui sua API

In [0]:
use_hf = True
sentiment_pipe = None

try:
    print("Carregando modelo de sentimento (isso pode levar alguns minutos na primeira vez)...")
    sentiment_pipe = pipeline(
        task='sentiment-analysis',
        model='cardiffnlp/twitter-xlm-roberta-base-sentiment',
        top_k=None,
        device=-1 # Força CPU para evitar erros em clusters sem GPU
    )
    print("Modelo carregado com sucesso!")
except Exception as err:
    use_hf = False
    print(f"Erro ao carregar modelo, usando fallback de regras. Erro: {err}")

Carregando modelo de sentimento (isso pode levar alguns minutos na primeira vez)...


config.json:   0%|          | 0.00/841 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-xlm-roberta-base-sentiment
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

Could not extract SentencePiece model from /home/spark-37f2e4a6-2e56-43b1-b2a3-09/.cache/huggingface/hub/models--cardiffnlp--twitter-xlm-roberta-base-sentiment/snapshots/f2f1202b1bdeb07342385c3f807f9c07cd8f5cf8/sentencepiece.bpe.model using sentencepiece library due to 
SentencePieceExtractor requires the SentencePiece library but it was not found in your environment. Check out the instructions on the
installation page of its repo: https://github.com/google/sentencepiece#installation and follow the ones
that match your environment. Please note that you may need to restart your runtime after installation.
. Falling back to TikToken extractor.


Erro ao carregar modelo, usando fallback de regras. Erro: `tiktoken` is required to read a `tiktoken` file. Install it with `pip install tiktoken`.


5: Funções de Lógica (Sentimento e Categoria)

In [0]:
POS_WORDS = ['bom','boa','otimo','ótimo','excelente','perfeito','obrigado','obrigada','show','resolvido','funcionou','sucesso','rapido','rápido']
NEG_WORDS = ['erro','bug','falha','caiu','fora do ar','instavel','instável','lento','lenta','demora','sem retorno','sem resposta','nao funciona','não funciona','problema','travando','trava','pessimo','péssimo','horrivel','horrível','inaceitavel','inaceitável','urgente','401','500','timeout','rejeitado','recusado']

def rule_sentiment(text_val):
    t = str(text_val).lower()
    pos_hits = sum([1 for w in POS_WORDS if w in t])
    neg_hits = sum([1 for w in NEG_WORDS if w in t])
    if neg_hits > pos_hits and neg_hits > 0: return 'NEGATIVA'
    if pos_hits > neg_hits and pos_hits > 0: return 'POSITIVA'
    return 'NEUTRA'

CAT_RULES = [
    ('INSTABILIDADE NA PLATAFORMA', ['fora do ar','instavel','instável','caiu','travando','trava','lento','lenta','timeout','latencia','latência','500','502','503']),
    ('API/AUTENTICACAO', ['api','token','401','403','autenticacao','autenticação','oauth','jwt']),
    ('BUGS', ['bug','erro','falha','stack','excecao','exceção','quebrou']),
    ('SUPORTE/ATENDIMENTO', ['sem retorno','sem resposta','demora','atraso','chamado','ticket','suporte','atendimento']),
    ('FINANCEIRO', ['fatura','cobranca','cobrança','boleto','pix','nota fiscal','nf','reembolso','pagamento','preco','preço','plano']),
    ('TREINAMENTO/ONBOARDING', ['treinamento','trilha','onboarding','curso','como usar','tutorial','documentacao','documentação']),
    ('FEATURE REQUEST', ['seria bom','poderia ter','faltou','queria','gostaria','sugestao','sugestão','melhoria','feature'])
]

def categorize(text_val):
    t = str(text_val).lower()
    for cat, kws in CAT_RULES:
        for kw in kws:
            if kw in t: return cat
    return 'FEEDBACK GERAL'

def hf_to_pt(label_val):
    mapping = {'LABEL_0': 'NEGATIVA', 'LABEL_1': 'NEUTRA', 'LABEL_2': 'POSITIVA'}
    return mapping.get(label_val, 'NEUTRA')

6: Execução do Processamento

In [0]:
texts = df['mensagem'].astype(str).tolist()
sent_out = []

if sentiment_pipe is not None:
    batch_size = 32 # Reduzido para estabilidade no Community Edition
    for start_idx in tqdm(range(0, len(texts), batch_size), desc="Classificando Sentimentos"):
        batch_txt = texts[start_idx:start_idx+batch_size]
        preds = sentiment_pipe(batch_txt)
        for item in preds:
            # Pega o label com maior score
            best = sorted(item, key=lambda x: x['score'], reverse=True)[0]
            sent_out.append(hf_to_pt(best['label']))
else:
    for t in tqdm(texts, desc="Processando via Regras"):
        sent_out.append(rule_sentiment(t))

df['sentimentalidade'] = sent_out
df['categoria'] = [categorize(x) for x in tqdm(texts, desc="Categorizando")]

print("Processamento concluído!")

Processando via Regras:   0%|          | 0/2200 [00:00<?, ?it/s]

Categorizando:   0%|          | 0/2200 [00:00<?, ?it/s]

Processamento concluído!


7: Salvamento no Volume

In [0]:
df.to_csv(output_path, index=False)
print(f"Sucesso! O arquivo processado foi salvo em: {output_path}")

Sucesso! O arquivo processado foi salvo em: /Volumes/workspace/voc/voc/voc_gold_processed_2025.csv
